# 4 — Feature engineering


## Imports


In [1]:
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from scipy.signal import resample
from scipy.stats import skew, kurtosis


## Configuration


In [2]:
from pathlib import Path
# ใช้ project root เดียวกัน ไม่ต้อง import โค้ด pipeline จาก program
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'dataset').is_dir() and (p / 'model').is_dir() and (p / 'program').is_dir())
RAW = ROOT / 'dataset/raw/mit-bih-arrhythmia-database-1.0.0'
DATA = ROOT / 'dataset/processed'
ARTIFACTS = ROOT / 'model/artifacts'
REPORTS = ROOT / 'model/reports'
CLASSES = ['N', 'S', 'V', 'F', 'Q']
SYMBOLS = ['NLRej', 'AaJS', 'VE', 'F', '/fQ']
LABEL_MAP = {s: i for i, group in enumerate(SYMBOLS) for s in group}
SEED = 42
PREPROCESSING = {'version': 1, 'fs': 360, 'left': 90, 'right': 162,
                 'band': [0.5, 40], 'order': 4, 'leads': ['MLII', 'V5']}

## Artifact helpers


In [3]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'ไม่พบ {path}; กรุณารัน notebook ขั้นก่อนหน้าก่อน')
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


## Define waveform and RR features


In [4]:
def features(waves, rr):
    delta = np.diff(waves, axis=1)
    stats = np.column_stack([
        waves.mean(1), waves.std(1), waves.min(1), waves.max(1),
        np.median(waves, axis=1), np.ptp(waves, axis=1),
        np.sqrt(np.mean(waves ** 2, axis=1)), np.sum(waves ** 2, axis=1),
        skew(waves, axis=1), kurtosis(waves, axis=1),
        delta.mean(1), delta.std(1), delta.min(1), delta.max(1)])
    names = ['rr_previous', 'rr_next', 'rr_ratio', 'rr_trailing_10',
             'mean', 'std', 'min', 'max', 'median', 'peak_to_peak', 'rms', 'energy',
             'skew', 'kurtosis', 'diff_mean', 'diff_std', 'diff_min', 'diff_max']
    names += [f'morphology_{i:02d}' for i in range(64)]
    values = np.column_stack([rr, stats, resample(waves, 64, axis=1)])
    values[~np.isfinite(values)] = np.nan
    frame = pd.DataFrame(values, columns=names)
    # indicator คงที่ทุกคอลัมน์ ป้องกัน schema เปลี่ยนเมื่อ missing ใน inference
    return pd.concat([frame, frame.isna().astype(float).add_suffix('_missing')], axis=1)

## Build features per partition


In [5]:
state = read_json(DATA / 'preprocessing.json')
frames = {}
for split in ['train', 'validation', 'test']:
    folder = DATA / split
    frames[split] = features(np.load(folder / 'waveforms.npy'), np.load(folder / 'rr.npy'))

## Fit imputer on Train


In [6]:
imputer = SimpleImputer(strategy='median', keep_empty_features=True)
imputer.fit(frames['train'])
joblib.dump(imputer, DATA / 'imputer.joblib')
state['feature_order'] = list(frames['train'].columns)
write_json(DATA / 'preprocessing.json', state)

## Transform and save


In [7]:
for split, frame in frames.items():
    folder = DATA / split
    frame.to_parquet(folder / 'features.parquet', index=False)
    np.save(folder / 'features.npy', imputer.transform(frame).astype(np.float32))
# แยก artifact ของ Test ไว้ต่างหาก โค้ดฝึกไม่โหลดไฟล์เหล่านี้
files = [p for p in DATA.rglob('*') if p.is_file() and p.name != 'prepared.json']
write_json(DATA / 'prepared.json', {'split_hash': state['split_hash'],
           'files': {p.relative_to(DATA).as_posix(): sha256(p) for p in files}})
{'feature_count': len(state['feature_order']), 'split_hash': state['split_hash']}

{'feature_count': 164,
 'split_hash': '2f039cab94163c262e74ef97f43ee25198316844c88295d6e152794e4a974743'}